# Metadata Ingestion & Filtering — Verification Notebook
Loads a dataset **with metadata** into Chroma, then verifies every metadata feature Chroma Studio relies on actually works: **ingestion, summary counts, equality filter, multi-value ($in) filter, and combined filters.**

Run this first — if every check here passes, the same operations will work in the Chroma Studio app (it uses the exact same Chroma calls under the hood).

## Setup
Uses your real in-house embedder. If you only want to test metadata plumbing without live endpoints, flip `USE_MOCK = True` below — metadata filtering doesn't depend on the embedder at all, so a mock is fine for testing the metadata path.

In [ ]:
USE_MOCK = True   # set False to use your real InHouseEmbeddings()

import chromadb, pandas as pd, numpy as np

if USE_MOCK:
    import re
    class MockEmbedder:
        def __call__(self, input):
            if isinstance(input, str): input = [input]
            out = []
            for t in input:
                v = np.zeros(64)
                for w in re.findall(r"[a-z0-9]+", t.lower()):
                    v[abs(hash(w)) % 64] += 1
                n = np.linalg.norm(v); out.append((v/n if n else v).tolist())
            return out
        def embed_documents(self, texts): return self(texts)
        def embed_query(self, text): return self([text])[0]
    embedder = MockEmbedder()
    print("Using MOCK embedder (fine for testing metadata — filtering is embedder-independent).")
else:
    from inhouse_wrappers import InHouseEmbeddings
    embedder = InHouseEmbeddings()
    print("Using real InHouseEmbeddings().")

## 1. Load the dataset and inspect its metadata columns

In [ ]:
df = pd.read_csv("dataset/support_docs.csv")
print("Rows:", len(df))
print("Columns:", list(df.columns))
# text goes in as the document; everything else (except id) is metadata
META_FIELDS = ["topic", "type", "product", "priority"]
df.head()

## 2. Ingest into Chroma WITH metadata
The key call: pass a `metadatas` list parallel to `ids`/`documents`, one dict per doc. This is exactly what Chroma Studio's Add tab does.

In [ ]:
client = chromadb.PersistentClient(path="./meta_test_db")   # local folder
# fresh start so re-runs are clean
try:
    client.delete_collection("support_docs")
except Exception:
    pass
coll = client.get_or_create_collection("support_docs", metadata={"hnsw:space": "cosine"})

ids = df["id"].tolist()
docs = df["text"].tolist()
metadatas = [{f: row[f] for f in META_FIELDS} for _, row in df.iterrows()]
embeddings = embedder.embed_documents(docs) if hasattr(embedder, "embed_documents") else embedder(docs)

coll.add(ids=ids, embeddings=embeddings, documents=docs, metadatas=metadatas)
print(f"Ingested {coll.count()} documents with metadata.")
print("Example stored metadata:", coll.get(ids=['fin_01'], include=['metadatas'])['metadatas'][0])

### ✅ Check 1 — metadata actually stored
The example above should print `{'topic': 'finance', 'type': 'core', 'product': 'payments', 'priority': 'high'}`. If metadata came back empty, ingestion dropped it — stop and check the `metadatas=` argument.

## 3. Metadata summary (what Studio's summary panel computes)

In [ ]:
def scan_metadata(coll, cap=2000):
    got = coll.get(limit=min(cap, coll.count()), include=["metadatas"])
    metas = got.get("metadatas") or []
    keys = sorted({k for m in metas if m for k in m.keys()})
    summary = {}
    for k in keys:
        counts = {}
        for m in metas:
            if m and k in m:
                counts[str(m[k])] = counts.get(str(m[k]), 0) + 1
        summary[k] = dict(sorted(counts.items(), key=lambda x: -x[1]))
    return keys, summary

keys, summary = scan_metadata(coll)
print("Metadata fields found:", keys)
for k, counts in summary.items():
    print(f"  {k}: {counts}")

### ✅ Check 2 — summary matches the dataset
Compare the printed counts to what `build_dataset.py` reported (topic: cooking 3, finance 4, shipping 3, api 3, account 3; type: core 12, near_duplicate 2, edge_case 2; etc.). They must match exactly — this is the panel Studio shows.

## 4. Equality filter — `where={field: value}`
This is what Studio's filter builds when you pick ONE value.

In [ ]:
r = coll.get(where={"topic": "finance"}, include=["documents", "metadatas"])
print(f"topic=finance -> {len(r['ids'])} docs:", sorted(r["ids"]))
assert len(r["ids"]) == 4, "expected 4 finance docs"

r = coll.get(where={"priority": "high"}, include=["documents"])
print(f"priority=high -> {len(r['ids'])} docs:", sorted(r["ids"]))
assert len(r["ids"]) == 5, "expected 5 high-priority docs"
print("\n✅ Check 3 — equality filters return the right counts.")

## 5. Multi-value filter — `where={field: {'$in': [...]}}`
What Studio builds when you keep SEVERAL values of a field.

In [ ]:
r = coll.get(where={"topic": {"$in": ["cooking", "api"]}}, include=["metadatas"])
print(f"topic in [cooking, api] -> {len(r['ids'])} docs:", sorted(r["ids"]))
assert len(r["ids"]) == 6, "expected 3 cooking + 3 api = 6"

r = coll.get(where={"type": {"$in": ["near_duplicate", "edge_case"]}}, include=["documents"])
print(f"type in [near_duplicate, edge_case] -> {len(r['ids'])} docs:", sorted(r["ids"]))
assert len(r["ids"]) == 4, "expected 2 + 2 = 4"
print("\n✅ Check 4 — $in multi-value filters work.")

## 6. Combined filter — multiple fields with `$and`
Chroma 1.5.9 combines conditions across fields with `$and`. Studio doesn't expose this in the UI yet, but it's good to know the data supports it.

In [ ]:
r = coll.get(
    where={"$and": [{"topic": "finance"}, {"priority": "high"}]},
    include=["documents", "metadatas"],
)
print(f"topic=finance AND priority=high -> {len(r['ids'])} docs:", sorted(r["ids"]))
for i, doc in zip(r["ids"], r["documents"]):
    print(f"   {i}: {doc[:55]}")
print("\n✅ Check 5 — combined multi-field filters work.")

## 7. Filtered visualize path — get embeddings for a metadata subset
This is exactly what Studio's Visualize tab does when a metadata filter is active: `coll.get(where=..., include=['embeddings', ...])`.

In [ ]:
r = coll.get(where={"product": "platform"},
             include=["embeddings", "documents", "metadatas"])
vecs = np.array(r["embeddings"])
print(f"product=platform -> {len(r['ids'])} docs, embeddings shape {vecs.shape}")
assert vecs.shape[0] == 9, "expected 9 platform docs"

# quick 2D projection to confirm the filtered subset is plottable
from sklearn.decomposition import PCA
coords = PCA(n_components=2).fit_transform(vecs)
print("PCA on filtered subset ->", coords.shape, "(ready to scatter in Studio)")
print("\n✅ Check 6 — the Visualize tab's filtered-embedding path works.")

## 8. All checks summary

In [ ]:
print("If you reached here with no AssertionError, ALL metadata paths work:")
print("  1. metadata stored on ingest")
print("  2. summary counts correct")
print("  3. equality filter (where={field:value})")
print("  4. multi-value filter (where={field:{'$in':[...]}})")
print("  5. combined multi-field filter ($and)")
print("  6. filtered embedding retrieval for Visualize")
print()
print("=> Chroma Studio's metadata summary, Browse filter, and Visualize filter")
print("   will all work on this collection. Point Studio's folder at:")
import os
print("  ", os.path.abspath("./meta_test_db"))
print("   and select the 'support_docs' collection.")

## Next: verify it visually in Chroma Studio
1. In Chroma Studio's sidebar, set the local folder to the `meta_test_db` path printed above, select `support_docs`.
2. **Browse tab** — open 'Metadata summary' (should match Check 2), then use the metadata filter to keep only `topic=finance` (should show 4 docs, matching Check 3).
3. **Visualize tab** — color by `topic`, then apply the metadata filter to `product=platform` (should plot 9 points, matching Check 6).
If the app shows the same numbers this notebook asserted, ingestion and filtering are confirmed working end to end.